In [1]:
import os, pathlib
import pandas as pd
import numpy as np
import re

In [2]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"
laws_de_path = "../data/laws_de.csv"
val_path = "../data/val.csv"
court_consdr = "../data/court_considerations.csv"

In [3]:
train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
law = pd.read_csv(laws_de_path)
court = pd.read_csv(court_consdr)

In [5]:
law.iloc[0]['text']

'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.\n Sie wird den Brunnen wie bis anhin mit Wasser versehen.\n Die Eidgenosse

In [24]:
def extract_article_structure(text: str) -> list[dict]:
    """
    Extract article structure from legal text.
    Returns: [{"article": "Art. 1", "abs": ["Abs. 1 content", "Abs. 2 content", ...]}]
    """
    # Pattern for Swiss legal articles: Art. 1, Art. 1 Abs. 1, Art. 1a, § 1, etc.
    article_pattern = r"(Art\.?\s+\d+[a-zA-Z]*)"
    abs_pattern = r"(Abs\.?\s+\d+)"
    
    articles = []
    current_article = None
    
    lines = text.split('\n')
    print(f"len = {len(lines)}")
    i = 0
    
    while i < len(lines):
        #print("matching")
        line = lines[i].strip()
        
        # Check for article header
        if re.search(article_pattern, line):
            article_match = re.search(article_pattern, line)
            print("matching")
            if article_match:
                # Save previous article if exists
                if current_article and current_article.get('content'):
                    articles.append(current_article)
                
                current_article = {
                    'article': article_match.group(1),
                    'content': line,
                    'sections': []
                }
        elif current_article and line:
            current_article['content'] += '\n' + line
        
        i += 1
    print(f"articles = \n{articles}")
    print(f"current_article = \n{current_article}")

extract_article_structure(law.iloc[0]['text'])
#print(articles)

len = 10
articles = 
[]
current_article = 
None


In [19]:
df = law[0:10]

In [20]:
type(df)

pandas.core.frame.DataFrame

In [21]:
df

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
5,Art. 4 Abs. 2 112,"2 Sie übernimmt auch die Verpflichtung, die er...",Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
6,Art. 4 Abs. 3 112,3 Im Fall die Schweizerische Eidgenossenschaft...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
7,Art. 5 Abs. 1 112,1 Sollte infolge förmlichen Beschlusses der ko...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
8,Art. 5 Abs. 2 112,2 Für den nämlichen Fall übernimmt die Schweiz...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
9,Art. 8 112,Infolge Übernahme der durch diese Übereinkunft...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [22]:
for idx, row in df.iterrows() :
    text = row.get('text', '')
    extract_article_structure(text=text)

articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None
articles = 
[]
current_article = 
None


In [ ]:

def split_by_sentences(self, text: str, article_header: str) -> List[Dict]:
    """Split text by sentences (German legal text patterns)."""
    chunks = []
    
    # German sentence enders: . ! ? followed by space and capital letter
    sentence_pattern = r'(?<=[.!?])\s+(?=[A-Z])'
    sentences = re.split(sentence_pattern, text)
    
    current_chunk = ""
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        test_chunk = current_chunk + " " + sentence if current_chunk else sentence
        tokens = TokenCounter.count_tokens(test_chunk)
        
        if tokens <= self.max_tokens:
            current_chunk = test_chunk
        else:
            # Save current chunk if it meets minimum
            if current_chunk and TokenCounter.count_tokens(current_chunk) >= self.min_tokens:
                chunks.append({
                    'text': current_chunk.strip(),
                    'tokens': TokenCounter.count_tokens(current_chunk),
                    'article': article_header,
                    'type': 'sentence_group'
                })
            
            # Start new chunk
            current_chunk = sentence
    
    # Don't forget last chunk
    if current_chunk and TokenCounter.count_tokens(current_chunk) >= self.min_tokens:
        chunks.append({
            'text': current_chunk.strip(),
            'tokens': TokenCounter.count_tokens(current_chunk),
            'article': article_header,
            'type': 'sentence_group'
        })
    
    return chunks

In [27]:
name = ""
title = "roy"
name + " " + title if name else title

'roy'

In [30]:
# With this:
law_df   = pd.read_parquet("../artifacts/chunks/laws_chunks.parquet")
court_df = pd.read_parquet("../artifacts/chunks/court_chunks.parquet")

In [31]:
law.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [32]:
law_df.head()

,source_type,citation,title,chunk_id,text,tokens,article,chunk_type,position
0,law,Art. 1 112,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,Art. 1 112_chunk_0,Die Einwohnergemeinde Bern tritt der Schweizer...,278.0,Art. 1 112,sentence_group,0
1,law,Art. 1 112,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,Art. 1 112_chunk_1,"Die Eidgenossenschaft verpflichtet sich, den F...",286.0,Art. 1 112,sentence_group,1
2,law,Art. 2 112,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,Art. 2 112_chunk_0,Die Einwohnergemeinde Bern wird ferner der Sch...,88.0,Art. 2 112,sentence_group,0
3,law,Art. 3 Abs. 1 112,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,Art. 3 Abs. 1 112_chunk_0,1 Falls die Schweizerische Eidgenossenschaft z...,242.0,Art. 3 Abs. 1 112,sentence_group,0
4,law,Art. 3 Abs. 1 112,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...,Art. 3 Abs. 1 112_chunk_1,Die Einwohnergemeinde ist jedoch zu einer solc...,60.0,Art. 3 Abs. 1 112,sentence_group,1


In [ ]:
law_df[""]

'Übereinkunft vom 22. Juni 1875 zwischen dem Schweizerischen Bundesrate und dem Einwohnergemeinderate der Stadt Bern betreffend die Leistungen der Stadt Bern an den Bundessitz'

In [2]:
df = pd.read_csv("../artifacts/submission.csv")

In [3]:
df

,query_id,citations
0,0,Art. 20 Abs. 3 131.211; BGE 139 I 16 E. 5; Art...
1,1,Art. 45 131.211; Art. 9 Abs. 1 112; Art. 135 A...
2,2,Art. 67 Abs. 2 131.211; Art. 13 Abs. 1 131.214...
3,3,BGE 139 I 16 E. 5; Art. 20 Abs. 3 131.211; Art...
4,4,Art. 48 131.211; Art. 9 Abs. 2 112; Art. 74 Ab...


In [5]:
df = pd.read_csv("../artifacts/pipeline_logs.csv")

In [6]:
df

,query_id,query,expanded,citations,n_results
0,0,May a court lawfully order a three‑month exten...,You are an expert Swiss lawyer. Extract the co...,Art. 20 Abs. 3 131.211; BGE 139 I 16 E. 5; Art...,10
1,1,A claimant holding a national vocational diplo...,You are an expert Swiss lawyer. Extract the co...,Art. 45 131.211; Art. 9 Abs. 1 112; Art. 135 A...,10
2,2,"A. Rivera, a Peruvian national born in 1994 an...",You are an expert Swiss lawyer. Extract the co...,Art. 67 Abs. 2 131.211; Art. 13 Abs. 1 131.214...,10
3,3,"Mr. Dalton, born in 1941 and resident in a sma...",You are an expert Swiss lawyer. Extract the co...,BGE 139 I 16 E. 5; Art. 20 Abs. 3 131.211; Art...,10
4,4,"A parent, separated from their co-parent since...",You are an expert Swiss lawyer. Extract the co...,Art. 48 131.211; Art. 9 Abs. 2 112; Art. 74 Ab...,10
